# Financial news voice assistant
## Solution overview

**FinMarket News Assistant** is a voice-enabled financial market intelligence agent that combines semantic search over a curated news database with high-level reasoning powered by Google's Gemini model.

The solution is built around three core components:

1. **Knowledge base** — ~5,500 financial news articles indexed in Elasticsearch, with vector embeddings generated automatically via the `semantic_text` field type using the `jina-embeddings-v5-text-small` model. This enables natural-language semantic search over article descriptions.

2. **Search tool** — An ES|QL query tool (`find_news_by_topic`) registered in the Elastic Agent Builder and exposed to the agent via the Elastic MCP server. It accepts a natural-language topic and optional date range, returning up to 50 ranked articles in a structured format.

3. **Voice agent** — A Python ADK agent running `gemini-2.5-flash-native-audio-preview-12-2025`, which interprets user queries, extracts search keywords, retrieves relevant articles via the MCP tool, and synthesizes a coherent, fact-grounded response.

The agent is deployed on **Google Cloud Run** with the ADK web UI, making it accessible via browser.

![News Agent Diagram](News%20Agent%20Diagram.jpg)

Deployment (GCP Cloud Runner): https://fin-news-agent-74205747062.us-central1.run.app/dev-ui/?app=fin_news_agent 
(underlying dataset covers only the year 2023)

## Financial Market News Database
### Prerequisites
To run this section of the notebook, you will need:
- Elasticsearch project (for example as an Elastic Cloud Serverless, Free trial is available or https://console.cloud.google.com/marketplace/product/elastic-prod/elastic-cloud)
- Sample dataset downloaded to the local folder: `polygon_news_sample.json` from https://www.kaggle.com/datasets/rdolphin/financial-news-with-ticker-level-sentiment/data

In [1]:
import os

# Elasticsearch 
es_url = input("Enter your Elasticsearch Endpoint URL (Example: https://my-elasticsearch-project-aa2287.es.us-central1.gcp.elastic.cloud:443): ")
# Kibana URL
kb_url = es_url.replace(".es.", ".kb.",1 )
# Elastic API Key
es_api_key = input("Enter your Elasticsearch API Key: ")

print("Using connection details:")
print(f"Elasticsearch URL: {es_url}")
print(f"Kibana URL: {kb_url}")
print(f"ES_API_KEY: ***************************{es_api_key[-5:]}")

Using connection details:
Elasticsearch URL: https://my-elasticsearch-project-aa2287.es.us-central1.gcp.elastic.cloud:443
Kibana URL: https://my-elasticsearch-project-aa2287.kb.us-central1.gcp.elastic.cloud:443
ES_API_KEY: ***************************1QQ==


In [2]:
# Connect and verify
from elasticsearch import Elasticsearch, helpers
es = Elasticsearch(es_url, api_key=es_api_key, request_timeout=60)
print(es.info()['version']['number'])

8.11.0


### Dataset
The source dataset (json) should be loaded to an Elastic index.

In [3]:
# Load data
# Data source: https://www.kaggle.com/datasets/rdolphin/financial-news-with-ticker-level-sentiment/data
import json
from pathlib import Path

FILE_PATH = Path('polygon_news_sample.json')

with FILE_PATH.open('r', encoding='utf-8') as f:
    data = json.load(f)

print(f"Loaded {len(data)} articles")

Loaded 5548 articles


The target data structure (Elastic Index mapping) includes the key source fields and an additional [semantic_text](https://www.elastic.co/docs/solutions/search/semantic-search/semantic-search-semantic-text) field, which automatically generates vector embeddings for semantic searches. [jina-embeddings-v5-text-small](https://www.elastic.co/search-labs/blog/jina-embeddings-v5-text) is available by default on Elastic Serverless, with no additional setup required.

In [4]:
# Create index: Define parameters
mapping = {
    'mappings': {
        'properties': {
            'id':            {'type': 'keyword'},
            'title':         {'type': 'text'},
            'published_utc': {'type': 'date'},
            'description':   {'type': 'text'},
            'description_semantic': {'type': 'semantic_text', 'inference_id': '.jina-embeddings-v5-text-small'}
        }
    }
}

INDEX_NAME = 'fin_market_news'

In [ ]:
# Create index
if es.indices.exists(index=INDEX_NAME): es.indices.delete(index=INDEX_NAME)

es.indices.create(index=INDEX_NAME, body=mapping)
print(f"Index '{INDEX_NAME}' created")

In [ ]:
# Bulk ingest
from elasticsearch.helpers import bulk

def generate_actions(data, index_name):
    for record in data:
        description = record.get('description') or ''
        yield {
            '_index': index_name,
            '_id':    record.get('id'),
            '_source': {
                'id':                   record.get('id'),
                'title':                record.get('title'),
                'published_utc':        record.get('published_utc'),
                'description':          description,
                'description_semantic': description   # raw text → Elastic embeds it
            }
        }

success, errors = bulk(es, generate_actions(data, INDEX_NAME), raise_on_error=False)

print(f"Indexed: {success} | Errors: {len(errors)}")

Indexed: 5548 | Errors: 0


Optionally, check that the semantic search works as expected:

In [5]:
# Check sematic search
def semantic_search(query_text, top_k=5):
    response = es.search(
        index=INDEX_NAME,
        body={
            'query': {
                'semantic': {
                    'field': 'description_semantic',
                    'query': query_text
                }
            },
            'size': top_k,
            '_source': ['title', 'description', 'published_utc']
        }
    )

    for hit in response['hits']['hits']:
        print(f"[{hit['_score']:.4f}] {hit['_source']['title']}")
        print(f"         {hit['_source']['description'][:120]}...")
        print()

semantic_search('Federal Reserve interest rate decision inflation')

[0.7976] Fed Meeting Live Updates: 10 Takeaways From May FOMC Meeting & Statement
         The Federal Reserve raised interest rates by 25 basis points, but uncertainty remains around future hikes due to the ban...

[0.7960] Inflation is Slowing, But is it Enough to Slow the Fed? 3 ETFs to Fight Back
         The article discusses the recent inflation data and its impact on the Federal Reserve's monetary policy. It suggests tha...

[0.7825] Initial Claims Come in Unchanged From Last Week
         The Federal Reserve kept interest rates unchanged at 5-5.25% but signaled more rate hikes are likely this year. The Euro...

[0.7822] The Zacks Analyst Blog Highlights Selective Insurance, Axos Financial, AssetMark Financial, East West Bancorp and State Street
         The article discusses how strong economic data, including improved retail sales and a robust labor market, have raised c...

[0.7702] 3 Top Stocks to Gain From Hawkish Fed Expectations
         Robust retail sales and job growth

## Elastic Agent Builder Tool Configuration

Detailed information about ES|QL Agent Builder Tools is available in [Elastic Docs](https://www.elastic.co/docs/explore-analyze/ai-features/agent-builder/tools/esql-tools).

For our project the semantic search tool is created using Elastic API:

In [ ]:
#Configure and create ESQL tool for financial news search

tool_id = "find_news_by_topic"

tool_definition = {
    "id": tool_id,
    "type": "esql",
    "description": """
    Search the database of financial market news by date (optional) and topic. 

    ## Usage
    Use this to find financial market news that correspond to vague user intent expressed in natural language 

    ## Parameters
    topic_subject_focus is a short natural language keyword string for semantic searching by descriptions
    If financial market news time interval is not fully specified, then date_after=1900-01-01T00:00:00.000Z and date_before=2100-01-01T00:00:00.000Z

    ## Results and Fallback
    If returned broad results that meet the requirements only partially and are not sufficiently focused on user intent, then it's necessary to analyze this output and apply additional filters by making additional reasoning steps without making additional database requests.
    """,
    "configuration": {
        "query": f"""
        FROM {INDEX_NAME} METADATA _id, _score
        | WHERE published_utc >= ?date_after AND published_utc <= ?date_before
        | WHERE MATCH(description_semantic, ?topic_subject_focus)
        | EVAL article = CONCAT(
            "[TITLE] ",       title,
            " | [DATE] ",     DATE_FORMAT("yyyy-MM-dd", published_utc),
            " | [SCORE] ",    TO_STRING(ROUND(_score, 3)),
            " | [SUMMARY] ",  description
            )
        | SORT _score DESC
        | KEEP article
        | LIMIT 50
      """,
        "params": {
            "date_after": {
                "type": "date",
                "optional": "true",
                "description": "Beginning of the time interval as an ISO datetime string (e.g. 2023-01-01T00:00:00.000Z). DEFAULT TO 1900-01-01T00:00:00.000Z"
            },
            "date_before": {
                "type": "date",
                "optional": "true",
                "description": "Ending of the time interval as an ISO datetime string (e.g. 2023-12-31T23:59:59.999Z). DEFAULT TO 2100-01-01T00:00:00.000Z"
            },
            "topic_subject_focus": {
                "type": "string",
                "description": "Key words for semantic search in financial news article descriptions."
            }
        },
    },
    "tags": ["finance", "news"],
}


In [ ]:
# Create ES|QL Tool.
import time, requests
from IPython.display import display, JSON, Markdown
HEADERS = {"Authorization": f"ApiKey {es_api_key}", "Content-Type": "application/json", "kbn-xsrf": "true"}

try:
    response = requests.post(
        f"{kb_url}/api/agent_builder/tools", headers=HEADERS, json=tool_definition
    )
    response.raise_for_status()

    print(f"✅ Successfully created the tool!")
    # Display the server's response

except requests.exceptions.RequestException as e:
    # Handle cases where the tool might already exist or other errors
    if (
        e.response.status_code == 400
        and "Tool with id" in e.response.text
        and "already exists" in e.response.text
    ):
        print(f"⚠️ Tool with ID '{tool_id}' already exists. Continuing.")
    else:
        print(f"❌ Tool creation failed: {e.response.text}")

✅ Successfully created the tool!


The tool is ready to be used by the ADK Agent via the [Elastic MCP server](https://www.elastic.co/search-labs/blog/elastic-mcp-server-agent-builder-tools). Also, it may be checked using the [MCP Inspector](https://modelcontextprotocol.io/docs/tools/inspector). 

## ADK Agent implementation
Agent creation involves the following steps:
- Prepare a Gemini API Key
- Create an agent template
- Overwrite the main agent file
- Append the required credentials to the .env file


In [17]:
# This step assumes that the Gemini API Key is available in OS Environment settings
import os
gemini_api_key = os.getenv("GEMINI_API_KEY")
print("Gemini API Key: " + gemini_api_key[:4] + "..." + gemini_api_key[-4:] if gemini_api_key else "Not set")


Gemini API Key: AIza...y4Jg


In [ ]:
!adk create --type=code fin_news_agent --model gemini-2.5-flash-native-audio-preview-12-2025 --api_key $gemini_api_key

In [ ]:
%%writefile fin_news_agent/agent.py

import os
import asyncio
if hasattr(asyncio, 'WindowsSelectorEventLoopPolicy'):
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())

from google.adk.agents.llm_agent import Agent
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPConnectionParams
from dotenv import load_dotenv
load_dotenv()

KIBANA_URL = os.getenv("KIBANA_URL")
KIBANA_API_KEY = os.getenv("KIBANA_API_KEY")

root_agent = Agent(
    model= 'gemini-2.5-flash-native-audio-preview-12-2025',
    name='news_agent',
    description='A helpful assistant for financial market news.',
    instruction="""
    **Your Identity:**
    You are a specialized Financial Market News analysis Assistant, designed to provide precise, data-driven insights from financial news articles.

    **Your Core Mission:**
    - Respond accurately and concisely to natural language queries from users seeking financial market intelligence.
    - Provide precise, objective, and actionable information derived solely from the tool at your disposal.
    - Generate your response in such a way that it should combine high-level overview of the available news with specific data.
    - Stay strictly on topic with financial market news queries.

    **Key Directives and Constraints:**
    -  **Strict Topic Mandate:** If a user asks about anything other than financial, economic or market news, you MUST refuse with the exact phrase: "Sorry, I can only help with financial news."
    -  **If a user's request is ambiguous, ask clarifying questions before proceeding.**
    -  **Information derived from the news articles must be naturally integrated into the response. Do not list facts one by one.**

    **Reasoning Framework:**
    1.  **Understand:** Deconstruct the user's query to understand the core intent. There may be several parameters and aspects of user intent. It's important to find each of them.
    2.  **Keywords Extraction:** Extract several keywords from the user question and use them to execute the `find_news_by_topic` tool.
    3.  **Retrieve:** Use the available `find_news_by_topic`tool to retrieve several relevant news articles.
    4.  **Synthesize:** Make sure that the retrieved news are relevant and sufficient to answer the question. Combine the information from all news articles into a single, comprehensive, and easy-to-understand answer. It should include:
    - High level summary overview of the key ideas from the most relevant news articles
    - List of specific relevant facts (dates, companies, events, products, etc.)

    Answer user questions about financial market news using the information retrieved from the find_news_by_topic tool. Extract several keywords from the user question and use them to execute the find_news_by_topic tool. Present the information in a clear and concise manner. If the user asks for something else, politely decline.
    """,
    tools=[
        McpToolset(
            connection_params=StreamableHTTPConnectionParams(
                url=f"{KIBANA_URL}/api/agent_builder/mcp",
                headers={"Authorization": f"ApiKey {KIBANA_API_KEY}"}
            ),
            tool_filter=['find_news_by_topic']
        )
    ],
)

Overwriting fin_news_agent/agent.py


In [ ]:
with open("fin_news_agent/.env", "a") as f:
    f.write(f"\nKIBANA_URL={kb_url}\n")
    f.write(f"KIBANA_API_KEY={es_api_key}\n")

Now, the agent can be launched locally:
```bash
adk web fin_news_agent          # browser UI
```

### Cloud Run Deployment
Before proceeding, make sure that your Google cloud project is set up.

In [19]:
gc_project = input("Enter your Google Cloud project ID (for example project-ed4bdfe6-4a39-433f-8c4): ")
print(f"Google Cloud project ID: {gc_project}")

Google Cloud project ID: project-ed4bdfe6-4a39-433f-8c4


In [ ]:
!gcloud config set project $gc_project

^C



To update your Application Default Credentials quota project, use the `gcloud auth application-default set-quota-project` command.
Project 'project-ed4bdfe6-4a39-433f-8c4' lacks an 'environment' tag. Please create or add a tag with key 'environment' and a value like 'Production', 'Development', 'Test', or 'Staging'. Add an 'environment' tag using `gcloud resource-manager tags bindings create`. See https://cloud.google.com/resource-manager/docs/creating-managing-projects#designate_project_environments_with_tags for details.
Updated property [core/project].


In [23]:
!echo y | adk deploy cloud_run --project=$gc_project --region=us-central1 --service_name=fin-news-agent --with_ui fin_news_agent

Start generating Cloud Run source files in C:\Users\a\AppData\Local\Temp\cloud_run_deploy_src\20260313_103753
Copying agent source code...
Copying agent source code completed.
Creating Dockerfile...
Creating Dockerfile complete: C:\Users\a\AppData\Local\Temp\cloud_run_deploy_src\20260313_103753\Dockerfile
Deploying to Cloud Run...
Cleaning up the temp folder: C:\Users\a\AppData\Local\Temp\cloud_run_deploy_src\20260313_103753


Allow unauthenticated invocations to [fin-news-agent] (y/N)?  
Building using Dockerfile and deploying container to Cloud Run service [fin-news-agent] in project [project-ed4bdfe6-4a39-433f-8c4] region [us-central1]
Building and deploying new service...
Validating configuration..............done
Uploading sources..............INFO: Not using ignore file.
.INFO: Uploading [C:\Users\a\AppData\Local\Temp\tmp5gikkt17\file.zip] to [run-sources-project-ed4bdfe6-4a39-433f-8c4-us-central1/services/fin-news-agent/1773383886.938053-1527f4d341d744169a8e80bbd1bcf100.zip]
......done
Building Container.....................................................................................................................................................................................................................................................................................................................................................................................................................

Start generating Cloud Run source files in C:\Users\a\AppData\Local\Temp\cloud_run_deploy_src\20260313_104012
Copying agent source code...
Copying agent source code completed.
Creating Dockerfile...
Creating Dockerfile complete: C:\Users\a\AppData\Local\Temp\cloud_run_deploy_src\20260313_104012\Dockerfile
Deploying to Cloud Run...
Cleaning up the temp folder: C:\Users\a\AppData\Local\Temp\cloud_run_deploy_src\20260313_104012


Building using Dockerfile and deploying container to Cloud Run service [fin-news-agent] in project [project-ed4bdfe6-4a39-433f-8c4] region [us-central1]
Building and deploying...
Validating configuration...............done
Uploading sources.............INFO: Not using ignore file.
.INFO: Uploading [C:\Users\a\AppData\Local\Temp\tmp9_5r6kpw\file.zip] to [run-sources-project-ed4bdfe6-4a39-433f-8c4-us-central1/services/fin-news-agent/1773384025.640288-c3c5d148912147e5b0e0d980c86a4b06.zip]
......done
Building Container................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................

In [24]:
## Set env vars
!gcloud run services update fin-news-agent --project={gc_project} --region=us-central1 --set-env-vars="GOOGLE_GENAI_USE_VERTEXAI=0,GOOGLE_API_KEY={gemini_api_key},KIBANA_URL={kb_url},KIBANA_API_KEY={es_api_key}"

Deploying...
Creating Revision..................................................................................................................................................................................................................................................................................................................done
Routing traffic.....done
Done.
Service [fin-news-agent] revision [fin-news-agent-00003-m86] has been deployed and is serving 100 percent of traffic.
Service URL: https://fin-news-agent-74205747062.us-central1.run.app
